In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [14]:
df = pd.read_csv('data/European_Bank.csv')

In [15]:
df.isna().sum()

Year               0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [16]:
df.duplicated().sum()

np.int64(0)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [18]:
df.nunique()

Year                   1
CustomerId         10000
Surname             2932
CreditScore          460
Geography              3
Gender                 2
Age                   70
Tenure                11
Balance             6382
NumOfProducts          4
HasCrCard              2
IsActiveMember         2
EstimatedSalary     9999
Exited                 2
dtype: int64

In [19]:
categorical_cols = df.select_dtypes(include='object').columns
categorical_cols

Index(['Surname', 'Geography', 'Gender'], dtype='object')

In [20]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
numerical_cols

Index(['Year', 'CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='object')

In [21]:
df.drop('Surname', axis=1, inplace=True)

In [22]:
print("Categories in 'Geography' variable:",end=" " )
print(df['Geography'].unique())

print("Categories in 'Gender' variable:     ",end=" " )
print(df['Gender'].unique())



Categories in 'Geography' variable: ['France' 'Spain' 'Germany']
Categories in 'Gender' variable:      ['Female' 'Male']


In [23]:
# define numerical & categorical columns
numeric_features = [feature for feature in df.columns if df[feature].dtype != 'O']
categorical_features = [feature for feature in df.columns if df[feature].dtype == 'O']

# print columns
print('We have {} numerical features : {}'.format(len(numeric_features), numeric_features))
print('\nWe have {} categorical features : {}'.format(len(categorical_features), categorical_features))

We have 11 numerical features : ['Year', 'CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']

We have 2 categorical features : ['Geography', 'Gender']


In [24]:
binary_cols = ["HasCrCard", "IsActiveMember", "Exited"]

for col in binary_cols:
    assert df[col].isin([0, 1]).all(), f"Non-binary values found in '{col}'"
 
    # No negative balances or salaries
    assert (df["Balance"] >= 0).all(), "Negative Balance detected"
    assert (df["EstimatedSalary"] > 0).all(), "Non-positive Salary detected"
 
    print(f"✅ Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print(f"   Churn Rate: {df['Exited'].mean():.2%}\n")
    

✅ Dataset loaded: 10,000 rows, 13 columns
   Churn Rate: 20.37%

✅ Dataset loaded: 10,000 rows, 13 columns
   Churn Rate: 20.37%

✅ Dataset loaded: 10,000 rows, 13 columns
   Churn Rate: 20.37%



ENGAGEMENT CLASSIFICATION FEATURES

In [25]:
conditions = [
        (df["IsActiveMember"] == 1) & (df["NumOfProducts"] >= 2),   # Active + multi-product
        (df["IsActiveMember"] == 0) & (df["NumOfProducts"] <= 1),   # Inactive + low-product
        (df["IsActiveMember"] == 1) & (df["NumOfProducts"] == 1),   # Active but low-product
        (df["IsActiveMember"] == 0) & (df["Balance"] > df["Balance"].median()),  # Inactive high-balance
    ]

labels = [
        "Active_Engaged",
        "Inactive_Disengaged",
        "Active_LowProduct",
        "Inactive_HighBalance",
    ]

df["EngagementSegment"] = np.select(conditions, labels, default="Other")
 
    # --- 1.2 Activity Score (weighted composite) ---
df["ActivityScore"] = (
        df["IsActiveMember"] * 3 +
        df["HasCrCard"] * 1 +
        (df["NumOfProducts"] >= 2).astype(int) * 2
    )
 
    # --- 1.3 Tenure Bands ---
df["TenureBand"] = pd.cut(
        df["Tenure"],
        bins=[-1, 1, 3, 6, 10],
        labels=["New (0-1y)", "Early (2-3y)", "Mid (4-6y)", "Loyal (7y+)"]
    )
 
    # --- 1.4 Is Long-Tenure Active ---
df["IsLongTenureActive"] = (
        (df["Tenure"] >= 5) & (df["IsActiveMember"] == 1)
    ).astype(int)

PRODUCT UTILIZATION FEATURES

In [28]:
df["ProductDepth"] = pd.cut(
        df["NumOfProducts"],
        bins=[0, 1, 2, 4],
        labels=["Single", "Dual", "Multi"]
    )
 
    # --- 2.2 Is Multi-Product Customer ---
df["IsMultiProduct"] = (df["NumOfProducts"] >= 2).astype(int)
 
    # --- 2.3 Product-Activity Interaction ---
    # Captures customers who are active AND using multiple products
df["ProductActivityIndex"] = df["NumOfProducts"] * df["IsActiveMember"]
 
    # --- 2.4 Card + Active Combo ---
df["CardActiveCombo"] = (
        (df["HasCrCard"] == 1) & (df["IsActiveMember"] == 1)
    ).astype(int)
 
    # --- 2.5 Product Engagement Ratio ---
    # Normalizes product count by max possible (4 products)
df["ProductEngagementRatio"] = df["NumOfProducts"] / 4.0
 
    # --- 2.6 Credit Card Stickiness Score ---
    # Rewards card ownership combined with activity
df["CreditCardStickinessScore"] = (
        df["HasCrCard"] * 0.4 +
        df["IsActiveMember"] * 0.4 +
        (df["NumOfProducts"] / 4) * 0.2
    )
 

FINANCIAL COMMITMENT & ENGAGEMENT FEATURES

In [30]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler


balance_median = df["Balance"].median()

salary_median  = df["EstimatedSalary"].median()
 
    # --- 3.1 Balance Tiers ---
df["BalanceTier"] = pd.cut(
        df["Balance"],
        bins=[-1, 1, 50_000, 100_000, 150_000, df["Balance"].max() + 1],
        labels=["Zero", "Low", "Mid", "High", "Premium"]
    )
 
    # --- 3.2 Zero Balance Flag ---
df["IsZeroBalance"] = (df["Balance"] == 0).astype(int)
 
    # --- 3.3 Balance-to-Salary Ratio ---
df["BalanceToSalaryRatio"] = df["Balance"] / (df["EstimatedSalary"] + 1)
 
    # --- 3.4 Salary–Balance Mismatch ---
    # High salary but near-zero balance → possible disengagement signal
df["SalaryBalanceMismatch"] = (
        (df["EstimatedSalary"] > salary_median) &
        (df["Balance"] < balance_median * 0.25)
    ).astype(int)
 
    # --- 3.5 At-Risk Premium Customer ---
    # High balance + inactive = dangerous churn segment
df["AtRiskPremiumCustomer"] = (
        (df["Balance"] > balance_median) &
        (df["IsActiveMember"] == 0)
    ).astype(int)
 
    # --- 3.6 High Balance Active (Sticky Premium) ---
df["HighBalanceActive"] = (
        (df["Balance"] > balance_median) &
        (df["IsActiveMember"] == 1)
    ).astype(int)
 
    # --- 3.7 Salary Tier ---
df["SalaryTier"] = pd.qcut(
        df["EstimatedSalary"],
        q=4,
        labels=["Q1_Low", "Q2_Mid", "Q3_Upper", "Q4_High"]
    )
 
    # --- 3.8 Wealth Index ---
scaler = MinMaxScaler()
df["WealthIndex"] = scaler.fit_transform(
        df[["Balance", "EstimatedSalary"]].values
    ).mean(axis=1)

DEMOGRAPHIC & BEHAVIORAL FEATURES

In [32]:
df["AgeBand"] = pd.cut(
        df["Age"],
        bins=[0, 25, 35, 45, 55, 65, 120],
        labels=["GenZ", "Millennial", "GenX_Early", "GenX_Late", "Boomer", "Senior"]
    )
 
    # --- 4.2 Senior Risk Flag (older customers churn differently) ---
df["IsSeniorRisk"] = (df["Age"] > 55).astype(int)
 
    # --- 4.3 Geography Encoding ---
geo_map = {"France": 0, "Spain": 1, "Germany": 2}
df["GeographyEncoded"] = df["Geography"].map(geo_map)
 
    # --- 4.4 Germany High-Balance Flag (Germany shows distinct churn patterns) ---
df["GermanyHighBalance"] = (
        (df["Geography"] == "Germany") &
        (df["Balance"] > df["Balance"].median())
    ).astype(int)
 
    # --- 4.5 Gender Encoded ---
df["GenderEncoded"] = LabelEncoder().fit_transform(df["Gender"])
 
    # --- 4.6 Credit Score Bands ---
df["CreditScoreBand"] = pd.cut(
        df["CreditScore"],
        bins=[0, 579, 669, 739, 799, 850],
        labels=["Poor", "Fair", "Good", "VeryGood", "Exceptional"]
    )
 
    # --- 4.7 Age × Tenure Interaction ---
df["AgeTenureProduct"] = df["Age"] * df["Tenure"]
 
    # --- 4.8 Young with Low Tenure (Early-Life Churn Risk) ---
df["YoungLowTenure"] = (
        (df["Age"] < 35) & (df["Tenure"] <= 2)
    ).astype(int)

KPI COMPOSITE SCORES

In [33]:
 # --- KPI 1: Engagement Retention Ratio Score ---
    # Proxy per customer: active + tenure contribution
df["EngagementRetentionScore"] = (
        df["IsActiveMember"] * 0.5 +
        (df["Tenure"] / df["Tenure"].max()) * 0.3 +
        df["HasCrCard"] * 0.2
    )
 
    # --- KPI 2: Product Depth Index ---
df["ProductDepthIndex"] = (
        (df["NumOfProducts"] / 4) * 0.6 +
        df["IsActiveMember"] * 0.4
    )
 
    # --- KPI 3: High-Balance Disengagement Rate (per customer flag) ---
df["HighBalanceDisengaged"] = (
        (df["Balance"] > df["Balance"].quantile(0.75)) &
        (df["IsActiveMember"] == 0)
    ).astype(int)
 
    # --- KPI 4: Credit Card Stickiness Score (already computed in step 2) ---
    # df["CreditCardStickinessScore"] ← already exists
 
    # --- KPI 5: Relationship Strength Index (RSI) ---
    # Holistic score combining engagement, products, tenure, and balance
balance_norm  = MinMaxScaler().fit_transform(df[["Balance"]]).flatten()
salary_norm   = MinMaxScaler().fit_transform(df[["EstimatedSalary"]]).flatten()
tenure_norm   = df["Tenure"] / df["Tenure"].max()
credit_norm   = (df["CreditScore"] - df["CreditScore"].min()) / \
                    (df["CreditScore"].max() - df["CreditScore"].min())
 
df["RelationshipStrengthIndex"] = (
        df["IsActiveMember"]          * 0.25 +
        df["ProductDepthIndex"]       * 0.25 +
        tenure_norm                   * 0.20 +
        balance_norm                  * 0.15 +
        credit_norm                   * 0.10 +
        df["HasCrCard"]               * 0.05
    )
 
    # --- Sticky Customer Flag (RSI threshold) ---
rsi_threshold = df["RelationshipStrengthIndex"].quantile(0.70)
df["IsStickyCustomer"] = (
        df["RelationshipStrengthIndex"] >= rsi_threshold
    ).astype(int)

RETENTION RISK SCORING

In [56]:
risk = pd.Series(np.zeros(len(df)), index=df.index)

risk += (df["IsActiveMember"] == 0).astype(int) * 2.0     # Inactive → +2
risk += (df["NumOfProducts"] == 1).astype(int) * 1.5      # Single product → +1.5
risk += (df["NumOfProducts"] > 2).astype(int) * 2.5       # 3–4 products → high churn (known pattern)
risk += df["IsZeroBalance"].astype(int) * 1.0             # Zero balance → +1
risk += df["AtRiskPremiumCustomer"].astype(int) * 2.0     # Premium + inactive → +2
risk += df["SalaryBalanceMismatch"].astype(int) * 1.0     # Mismatch → +1
risk += df["IsSeniorRisk"].astype(int) * 1.0              # Age > 55 → +1
risk += (df["Tenure"] <= 1).astype(int) * 1.0             # Very new → +1
risk += (df["CreditScore"] < 580).astype(int) * 0.5       # Poor credit → +0.5
risk += (df["GermanyHighBalance"]).astype(int) * 1.0      # Germany + high bal → +1
 
    # Normalize 0–10
df["RetentionRiskScore"] = (
        MinMaxScaler(feature_range=(0, 10))
        .fit_transform(risk.values.reshape(-1, 1))
        .flatten()
    )
 
df["RiskTier"] = pd.cut(
        df["RetentionRiskScore"],
        bins=[0, 3.33, 6.66, 10],
        labels=["Low Risk", "Medium Risk", "High Risk"],
        include_lowest=True
    )

In [37]:
pd.options.display.max_columns = 500
df

,Year,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,EngagementSegment,ActivityScore,TenureBand,IsLongTenureActive,ProductDepth,IsMultiProduct,ProductActivityIndex,CardActiveCombo,ProductEngagementRatio,CreditCardStickinessScore,BalanceTier,IsZeroBalance,BalanceToSalaryRatio,SalaryBalanceMismatch,AtRiskPremiumCustomer,HighBalanceActive,SalaryTier,WealthIndex,AgeBand,IsSeniorRisk,GeographyEncoded,GermanyHighBalance,GenderEncoded,CreditScoreBand,AgeTenureProduct,YoungLowTenure,EngagementRetentionScore,ProductDepthIndex,HighBalanceDisengaged,RelationshipStrengthIndex,IsStickyCustomer,RetentionRiskScore,RiskTier
0,2025,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1,Active_LowProduct,4,Early (2-3y),0,Single,0,1,1,0.25,0.85,Zero,1,0.000000,1,0,0,Q3_Upper,0.253367,GenX_Early,0,0,0,0,Fair,84,0,0.76,0.55,0,0.531300,0,3.684211,Medium Risk
1,2025,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,Active_LowProduct,3,New (0-1y),0,Single,0,1,0,0.25,0.45,Mid,0,0.744670,0,0,0,Q3_Upper,0.448370,GenX_Early,0,1,0,0,Fair,41,0,0.53,0.55,0,0.509205,0,2.631579,Low Risk
2,2025,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,Inactive_HighBalance,3,Loyal (7y+),0,Multi,1,0,0,0.75,0.55,Premium,0,1.401362,0,1,0,Q3_Upper,0.603006,GenX_Early,0,0,0,0,Poor,336,0,0.44,0.45,1,0.448354,0,7.368421,High Risk
3,2025,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0,Other,2,New (0-1y),0,Dual,1,0,0,0.50,0.10,Zero,1,0.000000,0,0,0,Q2_Mid,0.234560,GenX_Early,0,0,0,0,Good,39,0,0.03,0.30,0,0.164800,0,4.210526,Medium Risk
4,2025,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,Active_LowProduct,4,Early (2-3y),0,Single,0,1,1,0.25,0.85,High,0,1.587035,0,0,1,Q2_Mid,0.447823,GenX_Early,0,1,0,0,Exceptional,86,0,0.76,0.55,0,0.652537,1,1.578947,Low Risk
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2025,15606229,771,France,Male,39,5,0.00,2,1,0,96270.64,0,Other,3,Mid (4-6y),0,Dual,1,0,0,0.50,0.50,Zero,1,0.000000,0,0,0,Q2_Mid,0.240671,GenX_Early,0,0,0,1,VeryGood,195,0,0.35,0.30,0,0.309200,0,3.157895,Low Risk
9996,2025,15569892,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,Active_LowProduct,4,Loyal (7y+),1,Single,0,1,1,0.25,0.85,Mid,0,0.564102,0,0,0,Q3_Upper,0.368573,Millennial,0,0,0,1,Poor,350,0,1.00,0.55,0,0.704999,1,2.105263,Low Risk
9997,2025,15584532,709,France,Female,36,7,0.00,1,0,1,42085.58,1,Active_LowProduct,3,Loyal (7y+),1,Single,0,1,0,0.25,0.45,Zero,1,0.000000,0,0,0,Q1_Low,0.105195,GenX_Early,0,0,0,0,Good,252,0,0.71,0.55,0,0.599300,0,2.631579,Low Risk
9998,2025,15682355,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,Other,3,Early (2-3y),0,Dual,1,0,0,0.50,0.50,Mid,0,0.808222,0,0,0,Q2_Mid,0.381828,GenX_Early,0,2,0,1,VeryGood,126,0,0.29,0.30,0,0.314284,0,2.105263,Low Risk


In [38]:
df.shape

(10000, 46)

In [39]:
categorical_cols = df.select_dtypes(include='object').columns
categorical_cols

Index(['Geography', 'Gender', 'EngagementSegment'], dtype='object')

In [40]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
numerical_cols

Index(['Year', 'CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited', 'ActivityScore', 'IsLongTenureActive', 'IsMultiProduct',
       'ProductActivityIndex', 'CardActiveCombo', 'ProductEngagementRatio',
       'CreditCardStickinessScore', 'IsZeroBalance', 'BalanceToSalaryRatio',
       'SalaryBalanceMismatch', 'AtRiskPremiumCustomer', 'HighBalanceActive',
       'WealthIndex', 'IsSeniorRisk', 'GeographyEncoded', 'GermanyHighBalance',
       'GenderEncoded', 'AgeTenureProduct', 'YoungLowTenure',
       'EngagementRetentionScore', 'ProductDepthIndex',
       'HighBalanceDisengaged', 'RelationshipStrengthIndex',
       'IsStickyCustomer', 'RetentionRiskScore'],
      dtype='object')

In [42]:
# Binary : Gender , OverTime 

df['Gender']=df['Gender'].apply(lambda x: 1 if x == 'Male' else 0)

In [48]:
df

,Year,CustomerId,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,ActivityScore,TenureBand,IsLongTenureActive,ProductDepth,IsMultiProduct,ProductActivityIndex,CardActiveCombo,ProductEngagementRatio,CreditCardStickinessScore,BalanceTier,IsZeroBalance,BalanceToSalaryRatio,SalaryBalanceMismatch,AtRiskPremiumCustomer,HighBalanceActive,SalaryTier,WealthIndex,AgeBand,IsSeniorRisk,GeographyEncoded,GermanyHighBalance,GenderEncoded,CreditScoreBand,AgeTenureProduct,YoungLowTenure,EngagementRetentionScore,ProductDepthIndex,HighBalanceDisengaged,RelationshipStrengthIndex,IsStickyCustomer,RetentionRiskScore,RiskTier,France,Germany,Spain,Active_Engaged,Active_LowProduct,Inactive_Disengaged,Inactive_HighBalance,Other
0,2025,15634602,619,0,42,2,0.00,1,1,1,101348.88,1,4,Early (2-3y),0,Single,0,1,1,0.25,0.85,Zero,1,0.000000,1,0,0,Q3_Upper,0.253367,GenX_Early,0,0,0,0,Fair,84,0,0.76,0.55,0,0.531300,0,3.684211,Medium Risk,True,False,False,False,True,False,False,False
1,2025,15647311,608,0,41,1,83807.86,1,0,1,112542.58,0,3,New (0-1y),0,Single,0,1,0,0.25,0.45,Mid,0,0.744670,0,0,0,Q3_Upper,0.448370,GenX_Early,0,1,0,0,Fair,41,0,0.53,0.55,0,0.509205,0,2.631579,Low Risk,False,False,True,False,True,False,False,False
2,2025,15619304,502,0,42,8,159660.80,3,1,0,113931.57,1,3,Loyal (7y+),0,Multi,1,0,0,0.75,0.55,Premium,0,1.401362,0,1,0,Q3_Upper,0.603006,GenX_Early,0,0,0,0,Poor,336,0,0.44,0.45,1,0.448354,0,7.368421,High Risk,True,False,False,False,False,False,True,False
3,2025,15701354,699,0,39,1,0.00,2,0,0,93826.63,0,2,New (0-1y),0,Dual,1,0,0,0.50,0.10,Zero,1,0.000000,0,0,0,Q2_Mid,0.234560,GenX_Early,0,0,0,0,Good,39,0,0.03,0.30,0,0.164800,0,4.210526,Medium Risk,True,False,False,False,False,False,False,True
4,2025,15737888,850,0,43,2,125510.82,1,1,1,79084.10,0,4,Early (2-3y),0,Single,0,1,1,0.25,0.85,High,0,1.587035,0,0,1,Q2_Mid,0.447823,GenX_Early,0,1,0,0,Exceptional,86,0,0.76,0.55,0,0.652537,1,1.578947,Low Risk,False,False,True,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2025,15606229,771,1,39,5,0.00,2,1,0,96270.64,0,3,Mid (4-6y),0,Dual,1,0,0,0.50,0.50,Zero,1,0.000000,0,0,0,Q2_Mid,0.240671,GenX_Early,0,0,0,1,VeryGood,195,0,0.35,0.30,0,0.309200,0,3.157895,Low Risk,True,False,False,False,False,False,False,True
9996,2025,15569892,516,1,35,10,57369.61,1,1,1,101699.77,0,4,Loyal (7y+),1,Single,0,1,1,0.25,0.85,Mid,0,0.564102,0,0,0,Q3_Upper,0.368573,Millennial,0,0,0,1,Poor,350,0,1.00,0.55,0,0.704999,1,2.105263,Low Risk,True,False,False,False,True,False,False,False
9997,2025,15584532,709,0,36,7,0.00,1,0,1,42085.58,1,3,Loyal (7y+),1,Single,0,1,0,0.25,0.45,Zero,1,0.000000,0,0,0,Q1_Low,0.105195,GenX_Early,0,0,0,0,Good,252,0,0.71,0.55,0,0.599300,0,2.631579,Low Risk,True,False,False,False,True,False,False,False
9998,2025,15682355,772,1,42,3,75075.31,2,1,0,92888.52,1,3,Early (2-3y),0,Dual,1,0,0,0.50,0.50,Mid,0,0.808222,0,0,0,Q2_Mid,0.381828,GenX_Early,0,2,0,1,VeryGood,126,0,0.29,0.30,0,0.314284,0,2.105263,Low Risk,False,True,False,False,False,False,False,True


In [49]:
df = df.map(lambda x: 1 if x is True else 0 if x is False else x)

In [55]:
df

,Year,CustomerId,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,ActivityScore,TenureBand,IsLongTenureActive,ProductDepth,IsMultiProduct,ProductActivityIndex,CardActiveCombo,ProductEngagementRatio,CreditCardStickinessScore,BalanceTier,IsZeroBalance,BalanceToSalaryRatio,SalaryBalanceMismatch,AtRiskPremiumCustomer,HighBalanceActive,SalaryTier,WealthIndex,AgeBand,IsSeniorRisk,GeographyEncoded,GermanyHighBalance,GenderEncoded,CreditScoreBand,AgeTenureProduct,YoungLowTenure,EngagementRetentionScore,ProductDepthIndex,HighBalanceDisengaged,RelationshipStrengthIndex,IsStickyCustomer,RetentionRiskScore,RiskTier,France,Germany,Spain,Active_Engaged,Active_LowProduct,Inactive_Disengaged,Inactive_HighBalance,Other
0,2025,15634602,619,0,42,2,0.00,1,1,1,101348.88,1,4,Early (2-3y),0,Single,0,1,1,0.25,0.85,Zero,1,0.000000,1,0,0,Q3_Upper,0.253367,GenX_Early,0,0,0,0,Fair,84,0,0.76,0.55,0,0.531300,0,3.684211,Medium Risk,1,0,0,0,1,0,0,0
1,2025,15647311,608,0,41,1,83807.86,1,0,1,112542.58,0,3,New (0-1y),0,Single,0,1,0,0.25,0.45,Mid,0,0.744670,0,0,0,Q3_Upper,0.448370,GenX_Early,0,1,0,0,Fair,41,0,0.53,0.55,0,0.509205,0,2.631579,Low Risk,0,0,1,0,1,0,0,0
2,2025,15619304,502,0,42,8,159660.80,3,1,0,113931.57,1,3,Loyal (7y+),0,Multi,1,0,0,0.75,0.55,Premium,0,1.401362,0,1,0,Q3_Upper,0.603006,GenX_Early,0,0,0,0,Poor,336,0,0.44,0.45,1,0.448354,0,7.368421,High Risk,1,0,0,0,0,0,1,0
3,2025,15701354,699,0,39,1,0.00,2,0,0,93826.63,0,2,New (0-1y),0,Dual,1,0,0,0.50,0.10,Zero,1,0.000000,0,0,0,Q2_Mid,0.234560,GenX_Early,0,0,0,0,Good,39,0,0.03,0.30,0,0.164800,0,4.210526,Medium Risk,1,0,0,0,0,0,0,1
4,2025,15737888,850,0,43,2,125510.82,1,1,1,79084.10,0,4,Early (2-3y),0,Single,0,1,1,0.25,0.85,High,0,1.587035,0,0,1,Q2_Mid,0.447823,GenX_Early,0,1,0,0,Exceptional,86,0,0.76,0.55,0,0.652537,1,1.578947,Low Risk,0,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2025,15606229,771,1,39,5,0.00,2,1,0,96270.64,0,3,Mid (4-6y),0,Dual,1,0,0,0.50,0.50,Zero,1,0.000000,0,0,0,Q2_Mid,0.240671,GenX_Early,0,0,0,1,VeryGood,195,0,0.35,0.30,0,0.309200,0,3.157895,Low Risk,1,0,0,0,0,0,0,1
9996,2025,15569892,516,1,35,10,57369.61,1,1,1,101699.77,0,4,Loyal (7y+),1,Single,0,1,1,0.25,0.85,Mid,0,0.564102,0,0,0,Q3_Upper,0.368573,Millennial,0,0,0,1,Poor,350,0,1.00,0.55,0,0.704999,1,2.105263,Low Risk,1,0,0,0,1,0,0,0
9997,2025,15584532,709,0,36,7,0.00,1,0,1,42085.58,1,3,Loyal (7y+),1,Single,0,1,0,0.25,0.45,Zero,1,0.000000,0,0,0,Q1_Low,0.105195,GenX_Early,0,0,0,0,Good,252,0,0.71,0.55,0,0.599300,0,2.631579,Low Risk,1,0,0,0,1,0,0,0
9998,2025,15682355,772,1,42,3,75075.31,2,1,0,92888.52,1,3,Early (2-3y),0,Dual,1,0,0,0.50,0.50,Mid,0,0.808222,0,0,0,Q2_Mid,0.381828,GenX_Early,0,2,0,1,VeryGood,126,0,0.29,0.30,0,0.314284,0,2.105263,Low Risk,0,1,0,0,0,0,0,1
